### 4.2.2 工具的定义与实现

In [1]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from typing import List, Dict

为了让代码结构更清晰、更易于复用，我们来定义一个专属的LLM客户端类。这个类将封装所有与模型服务交互的细节，让我们的主逻辑可以更专注于智能体的构建。

In [2]:
# 加载 .env 文件中的环境变量
load_dotenv()

class HelloAgentsLLM:
    """
    为本书 "Hello Agents" 定制的LLM客户端。
    它用于调用任何兼容OpenAI接口的服务，并默认使用流式响应。
    """
    def __init__(self, model: str = None, apiKey: str = None, baseUrl: str = None, timeout: int = None):
        """
        初始化客户端。优先使用传入参数，如果未提供，则从环境变量加载。
        """
        self.model = model or os.getenv("LLM_MODEL_ID")
        apiKey = apiKey or os.getenv("LLM_API_KEY")
        baseUrl = baseUrl or os.getenv("LLM_BASE_URL")
        timeout = timeout or int(os.getenv("LLM_TIMEOUT", 60))
        
        if not all([self.model, apiKey, baseUrl]):
            raise ValueError("模型ID、API密钥和服务地址必须被提供或在.env文件中定义。")

        self.client = OpenAI(api_key=apiKey, base_url=baseUrl, timeout=timeout)

    def think(self, messages: List[Dict[str, str]], temperature: float = 0) -> str:
        """
        调用大语言模型进行思考，并返回其响应。
        """
        print(f"🧠 正在调用 {self.model} 模型...")
        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=messages,
                temperature=temperature,
                stream=True,
            )
            
            # 处理流式响应
            print("✅ 大语言模型响应成功:")
            collected_content = []
            for chunk in response:
                if not chunk.choices:
                    continue
                content = chunk.choices[0].delta.content or ""
                print(content, end="", flush=True)
                collected_content.append(content)
            print()  # 在流式输出结束后换行
            return "".join(collected_content)

        except Exception as e:
            print(f"❌ 调用LLM API时发生错误: {e}")
            return None

In [3]:
# --- 客户端使用示例 ---
if __name__ == '__main__':
    try:
        llmClient = HelloAgentsLLM()
        
        exampleMessages = [
            {"role": "system", "content": "You are a helpful assistant that writes Python code."},
            {"role": "user", "content": "写一个快速排序算法"}
        ]
        
        print("--- 调用LLM ---")
        responseText = llmClient.think(exampleMessages)
        if responseText:
            print("\n\n--- 完整模型响应 ---")
            print(responseText)

    except ValueError as e:
        print(e)


--- 调用LLM ---
🧠 正在调用 qwen3.6:35b 模型...
✅ 大语言模型响应成功:
这里提供两种常见的 Python 实现方式：**原地排序版（推荐，空间效率高）** 和 **简洁函数式版（代码短，易读）**。

 🔪 版本一：原地排序版（标准算法实现）
面试和工程中最常用的写法，通过索引操作直接在原数组上交换元素，额外空间复杂度仅为递归栈的 `O(log n)`。

python
 quick_sort(arr: list) -> list:
快速排序（原地排序）"""
    
 _quick_sort(low: int, high: int):
 low < high:
 分区操作，返回基准元素的最终索引位置
 = partition(low, high)
 递归排序左半部分和右半部分
quick_sort(low, pi - 1)
quick_sort(pi + 1, high)

 partition(low: int, high: int) -> int:
 = arr[high]  # 选择最后一个元素作为基准
 = low - 1        # i 指向小于基准的区域的末尾
        
 j in range(low, high):
 arr[j] <= pivot:
 += 1           i
[i], arr[j] = arr[j], arr[i]  # 将小于等于基准的元素交换到左侧
                
 将基准元素放到正确的位置（i+1）
[i + 1], arr[high] = arr[high], arr[i + 1]
 i + 1  return

quick_sort(0, len(arr) - 1)
 arr  # 原地修改，返回原列表方便链式调用
```

 🐍 版本二：Pythonic 简洁版（适合快速脚本）
列表推导式，代码极其简短，但会创建新列表，空间复杂度为 `O(n)`。

python
 quick_sort_pythonic(arr: list) -> list:
 len(arr) <= 1:
 arr    return
    
 = arr[len(arr) // 2]
  = [x for x in arr if x < pivot]
   = [x fo

针对本节设定的目标——回答关于“华为最新手机”的问题，我们需要为智能体提供一个网页搜索工具。在这里我们选用 SerpApi，它通过API提供结构化的Google搜索结果，能直接返回“答案摘要框”或精确的知识图谱信息。

首先，需要安装该库：

pip install google-search-results

我们的第一个工具是 search 函数，它的作用是接收一个查询字符串，然后返回搜索结果。

In [10]:
from serpapi import SerpApiClient

def search(query: str) -> str:
    """
    一个基于SerpApi的实战网页搜索引擎工具。
    它会智能地解析搜索结果，优先返回直接答案或知识图谱信息。
    """
    print(f"🔍 正在执行 [SerpApi] 网页搜索: {query}")
    try:
        api_key = os.getenv("SERPAPI_API_KEY")
        if not api_key:
            return "错误:SERPAPI_API_KEY 未在 .env 文件中配置。"

        params = {
            "engine": "google",
            "q": query,
            "api_key": api_key,
            "gl": "cn",  # 国家代码
            "hl": "zh-cn", # 语言代码
        }
        
        client = SerpApiClient(params)
        results = client.get_dict()
        
        # 智能解析:优先寻找最直接的答案
        if "answer_box_list" in results:
            return "\n".join(results["answer_box_list"])
        if "answer_box" in results and "answer" in results["answer_box"]:
            return results["answer_box"]["answer"]
        if "knowledge_graph" in results and "description" in results["knowledge_graph"]:
            return results["knowledge_graph"]["description"]
        if "organic_results" in results and results["organic_results"]:
            # 如果没有直接答案，则返回前三个有机结果的摘要
            snippets = [
                f"[{i+1}] {res.get('title', '')}\n{res.get('snippet', '')}"
                for i, res in enumerate(results["organic_results"][:3])
            ]
            return "\n\n".join(snippets)
        
        return f"对不起，没有找到关于 '{query}' 的信息。"

    except Exception as e:
        return f"搜索时发生错误: {e}"

当智能体需要使用多种工具时（例如，除了搜索，还可能需要计算、查询数据库等），我们需要一个统一的管理器来注册和调度这些工具。为此，我们创建一个 ToolExecutor 类。

In [11]:
from typing import Dict, Any

class ToolExecutor:
    """
    一个工具执行器，负责管理和执行工具。
    """
    def __init__(self):
        self.tools: Dict[str, Dict[str, Any]] = {}

    def registerTool(self, name: str, description: str, func: callable):
        """
        向工具箱中注册一个新工具。
        """
        if name in self.tools:
            print(f"警告:工具 '{name}' 已存在，将被覆盖。")
        self.tools[name] = {"description": description, "func": func}
        print(f"工具 '{name}' 已注册。")

    def getTool(self, name: str) -> callable:
        """
        根据名称获取一个工具的执行函数。
        """
        return self.tools.get(name, {}).get("func")

    def getAvailableTools(self) -> str:
        """
        获取所有可用工具的格式化描述字符串。
        """
        return "\n".join([
            f"- {name}: {info['description']}" 
            for name, info in self.tools.items()
        ])

现在，我们将 search 工具注册到 ToolExecutor 中，并模拟一次调用，以验证整个流程是否正常工作。

In [13]:
# --- 工具初始化与使用示例 ---
if __name__ == '__main__':
    # 1. 初始化工具执行器
    toolExecutor = ToolExecutor()

    # 2. 注册我们的实战搜索工具
    search_description = "一个网页搜索引擎。当你需要回答关于时事、事实以及在你的知识库中找不到的信息时，应使用此工具。"
    toolExecutor.registerTool("Search", search_description, search)
    
    # 3. 打印可用的工具
    print("\n--- 可用的工具 ---")
    print(toolExecutor.getAvailableTools())

    # 4. 智能体的Action调用，这次我们问一个实时性的问题
    print("\n--- 执行 Action: Search['英伟达最新的GPU型号是什么'] ---")
    tool_name = "Search"
    tool_input = "英伟达最新的GPU型号是什么"

    tool_function = toolExecutor.getTool(tool_name)
    if tool_function:
        observation = tool_function(tool_input)
        print("--- 观察 (Observation) ---")
        print(observation)
    else:
        print(f"错误:未找到名为 '{tool_name}' 的工具。")
        

工具 'Search' 已注册。

--- 可用的工具 ---
- Search: 一个网页搜索引擎。当你需要回答关于时事、事实以及在你的知识库中找不到的信息时，应使用此工具。

--- 执行 Action: Search['英伟达最新的GPU型号是什么'] ---
🔍 正在执行 [SerpApi] 网页搜索: 英伟达最新的GPU型号是什么
--- 观察 (Observation) ---
[1] 英伟达（NVIDIA）显卡全系列深度解析：从消费级到数据中心 ...
每两年一个架构大版本，是NVIDIA 近十年消费级GPU 的基本节奏。 2.2 RTX 50 系列（Blackwell 架构，2025）. 架构亮点：. 基于TSMC 4N 制程的Blackwell ...

[2] 比較最新系列與前一系列的GeForce 顯示卡
比較最新的RTX 30 系列與之前的RTX 20 系列、GTX 10 和900 系列顯示卡。找出規格、功能、支援的技術等。

[3] 英伟达产品系列分类、架构、数据中心GPU所有型号及参数 ...
H100 NVL GPU是英伟达公司的下一代加速计算平台，采用了全新的Hopper架构。该GPU支持PCIe 5.0标准，具有40Tb/s的IO带宽，可承载全球互联网的流量。
